In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata, describe, mode, fisher_exact
import scipy as sp
import scipy.stats as stats
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.metrics import pairwise_distances
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot
from wordcloud import WordCloud
from sklearn.decomposition import NMF, non_negative_factorization
from collections import Counter
from scipy.stats import hypergeom
from scipy.stats.mstats import gmean
from scipy.cluster import hierarchy
import squarify
try:
    import piplite
    await piplite.install(['ipywidgets'])
except ImportError:
    pass
import ipywidgets as widgets
import json

today = date.today()

**building 1kb regions**

In [3]:
chrm_lengths = [248956422, 242193529, 198295559, 190214555, 181538259, 170805979, 159345973, 145138636, 138394717, 133797422, 135086622, 133275309, 114364328, 107043718, 101991189, 90338345, 83257441, 80373285, 58617616, 64444167, 46709983, 50818468, 156040895, 57227415]
chrms = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', 'X', 'Y']

In [12]:
# out_f = '/home/ctrader/region_similarity/data/rank_based/hg38_chromosome_lengths_CT20230327.tsv'
# np.savetxt(out_f, np.vstack((chrms, chrm_lengths)).T, fmt='%s', delimiter='\t')

In [24]:
chrom_len_path = '/home/ctrader/region_similarity/data/rank_based/hg38_chromosome_lengths_CT20230327.tsv'

# load lengths of chromosomes
chrm_lengths = []
chrms = []
f = open(chrom_len_path, 'r')
for line in f.readlines():
    line = line.strip().split('\t')
    chrms.append(line[0])
    chrm_lengths.append(int(line[1]))
f.close()

In [18]:
scale = 2000
bw = 1000

# create regions
bed_out = []
for c, l in zip(chrms, chrm_lengths):
    start = 0
    end = scale
    seqname = 'chr' + c
    while start < l:
        bed_out.append([seqname, start, end])
        start += bw
        end += bw
bed_out = np.array(bed_out)

In [49]:
out_path = '/home/ctrader/region_similarity/data/rank_based/hg38.1kb.regions.CT20230323.bed'

# save
np.savetxt(out_path, bed_out, fmt='%s', delimiter='\t')

**combining datasets and reducing precision**

In [88]:
def load_per(per_f):
    hdf5_f = h5py.File(per_f, 'r')
    percentile = hdf5_f['percentile'][:]
    factors = np.char.decode(hdf5_f['factors'][:])
    color = np.char.decode(hdf5_f['colors'][:])
    hdf5_f.close()
    return percentile, factors, color

In [89]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
DHS_per_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank_percentile.hdf5'
epi_per_f = datadir + 'regions_1kb_CT20230323.epilogos.statesum_rank_percentile.hdf5'

# load region percentiles
DHS_per, DHS_factors, DHS_color = load_per(DHS_per_f)
epi_per, epi_factors, epi_color = load_per(epi_per_f)

In [90]:
percents = np.hstack((DHS_per, epi_per))
factor_order = np.append(DHS_factors, epi_factors)
metric_order = np.append(np.array(['DHSs'] * len(DHS_factors)),
                         np.array(['chromatin states'] *
                                  len(epi_factors)))
colors = np.append(DHS_color, epi_color)

In [91]:
percents.shape

(3088281, 34)

In [92]:
# datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
# out_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'
# print(out_f)
# np.save(out_f, percents.astype('float32'))

/home/ctrader/AG/NarrativeGenomics/region_narration/data/regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy


In [100]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
per_f = datadir + 'regions_1kb_CT20230323.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.npy'

t = time.time()
percents = np.load(per_f)
time.time() - t

1.3436346054077148

In [96]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
regions_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed'

f = open(regions_path, 'r')
regions_sep = []
for line in f.readlines():
    line = line.strip().split('\t')
    regions_sep.append(line)
f.close()
regions_sep = np.array(regions_sep)

In [97]:
regions_sep.shape

(3088281, 3)

In [98]:
# datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
# out_f = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed.npy'

# print(out_f)
# np.save(out_f, regions_sep)

/home/ctrader/AG/NarrativeGenomics/region_narration/data/hg38.1kb.regions.CT20230323.sorted.bed.npy


In [99]:
datadir = '/home/ctrader/AG/NarrativeGenomics/region_narration/data/'
regions_path = datadir + 'hg38.1kb.regions.CT20230323.sorted.bed.npy'

t = time.time()
regions_sep = np.load(regions_path)
time.time() - t

0.8398733139038086

**Limiting Search Space**

In [109]:
datadir = '/home/ctrader/region_similarity/data/rank_based/'
rank_file = datadir + 'regions_2kb_CT20230327.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.hdf5'

datadir = '/home/ctrader/region_similarity/data/rank_based/'
out_path = datadir + 'regions_2kb_CT20230327.DHS.factornormNMF.sum_enrich_corank.epilogos.statesum_rank.percentiles.limited_search_space.npz'

thresh = 0.95

In [110]:
hdf5_f = h5py.File(rank_file, 'r')
percents = hdf5_f['percentiles'][:]
regions = np.char.decode(hdf5_f['coordinates'][:])
hdf5_f.close()

In [111]:
inds_set = set()
for i in range(percents.shape[1]):
    inds = np.where(percents[:, i] >= thresh)[0]
    inds_set.update(inds)

In [112]:
top_inds = np.sort(list(inds_set))
top_percents = percents[top_inds]
top_regions = regions[top_inds]

# save
np.savez(out_path, percentiles=top_percents, coordinates=top_regions, original_inds=top_inds)